In [26]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.optimize as opt
from scipy.integrate import *
import torch

In [6]:
from bh_horizons import *

In [32]:
def P(V, k):
    """Polynomial that vanishes up to k-th derivative at V=0 and V=1."""
    return V**(k+1) * (1 - V)**(k+1)

def dP(V, k):
    """Derivative of the polynomial P(V, k)."""
    return (k+1) * V**k * (1 - V)**(k+1) - (k+1) * V**(k+1) * (1 - V)**k

In [28]:
def rho_drho(V, theta, k):
    """Neural network Ansatz for rho and its derivative."""
    p = (len(theta) - 1) // 3
    a0 = theta[0]
    a = theta[1:p+1]
    w = theta[p+1:2*p+1]
    b = theta[2*p+1:]
    
    # Neural network part
    V = np.asarray(V, dtype=float)
    z = V[..., None] * w + b
    NN = a0 + np.sum(a * np.tanh(z), axis=-1)
    dNN = np.sum(a * w * (1 - np.tanh(z)**2), axis=-1)
    
    # Polynomial part
    Pk, dPk = P(V, k), dP(V, k)
    
    rho = Pk * NN
    drho_dV = dPk * NN + Pk * dNN
    
    return rho, drho_dV

In [24]:
def xi_integral(theta, k):
    def f(V, y):
        xi, xip = y
        rho, drho = rho_drho(V, theta, k=k)  
        return [xip, -xi * (rho**2 + drho**2)]

    V_grid = np.linspace(0, 1, 5001)
    sol = solve_ivp(
        f,
        t_span=(1.0, 0.0),       # backwards
        y0=[1, 0.0],        # xi(1)=xi_+, xi'(1)=0
        t_eval=V_grid[::-1],
        method='DOP853',
        rtol=1e-10,
        atol=1e-12
        )
    xi_vals, xip_vals = sol.y[0][::-1], sol.y[1][::-1]
    I = simpson(xi_vals**2 * rho_drho(V_grid, theta, k=k)[0]**2, V_grid)
    
    return xi_vals, xip_vals, I


In [34]:
def sup_rU_for_q_lambda_massive(
    theta,
    k,
    xi_vals,
    xip_vals,
    q,
    lam,
    m_over_e=0.0,
    eM=None,
):
    V_grid = np.linspace(0.0, 1.0, len(xi_vals))

    # Final black-hole horizon radius and target charge
    r_plus, Qtarget = rplus_and_Qtarget(q, lam)

    # Physical r and r_V on C
    r_vals = r_plus * xi_vals
    rp_vals = r_plus * xip_vals

    # Scalar profile rho = alpha chi
    rho_vals, drho_vals = rho_drho(V_grid, theta, k)

    # I(alpha) = integral xi^2 rho^2 dV
    integrand_I = xi_vals**2 * rho_vals**2

    # Cumulative charge integral
    cum_I = cumulative_simpson(
        integrand_I,
        x=V_grid,
        initial=0.0
    )
    I_a = cum_I[-1]

    # Determine e.
    # Since M=1, eM = e.
    if eM is None:
        e = Qtarget / (r_plus**2 * I_a)
    else:
        e = eM

    # Scalar mass m = (m/e) * e
    m = m_over_e * e

    # Charge profile.
    # If eM is None, normalize to hit Qtarget exactly.
    # If eM is given, use the physical Maxwell integral.
    if eM is None:
        Q_vals = Qtarget * cum_I / cum_I[-1]
    else:
        Q_vals = e * r_plus**2 * cum_I

    charge_residual = Q_vals[-1] - Qtarget

    # Initial condition from zero Hawking mass at V=0:
    # r_U(0) = (Lambda r(0)^2/3 - 1)/(4 r_V(0))
    rU0 = (lam * r_vals[0]**2 / 3.0 - 1.0) / (4.0 * rp_vals[0])

    # Integrated equation:
    #
    # (r r_U)' =
    #   -1/4
    #   + Q^2/(4 r^2)
    #   + Lambda r^2/4
    #   + m^2 r^2 |Phi|^2/4
    #
    # and |Phi|^2 = rho^2.
    integrand_rU = (
        -0.25
        + Q_vals**2 / (4.0 * r_vals**2)
        + lam * r_vals**2 / 4.0
        + m**2 * r_vals**2 * rho_vals**2 / 4.0
    )

    r_rU_vals = (
        r_vals[0] * rU0
        + cumulative_simpson(
            integrand_rU,
            x=V_grid,
            initial=0.0
        )
    )

    rU_vals = r_rU_vals / r_vals

    return {
        "sup_rU": np.max(rU_vals),
        "rU_vals": rU_vals,
        "Q_vals": Q_vals,
        "r_plus": r_plus,
        "Qtarget": Qtarget,
        "I_theta": I_a,
        "e": e,
        "m": m,
        "m_over_e": m_over_e,
        "charge_residual": charge_residual,
    }

In [22]:
def penalty_xi(xi_vals):
    return np.mean(np.exp(-10.0 * xi_vals))


def penalty_rU(rU_vals):
    return np.mean(np.exp(10.0 * rU_vals))


def loss_C0(theta, eM, q, lam=0.0, w_Q=1.0, w_penalty=1.0):
    xi_vals, xip_vals, _ = xi_integral(theta, k=0)

    inf_xi = np.min(xi_vals)

    # Avoid calculating r_U when r is zero or negative.
    if inf_xi <= 1e-8 or not np.all(np.isfinite(xi_vals)):
        return 1e6 + 1e6 * (1e-8 - inf_xi)**2

    out = sup_rU_for_q_lambda_massive(
        theta=theta,
        k=0,
        xi_vals=xi_vals,
        xip_vals=xip_vals,
        q=q,
        lam=lam,
        m_over_e=0.0,
        eM=eM,
    )

    residual = out["charge_residual"]

    penalty = w_penalty * (
        penalty_xi(xi_vals)
        + penalty_rU(out["rU_vals"])
    )

    return w_Q * residual**2 + penalty

In [11]:
def init_theta(p, seed):
    rng = np.random.default_rng(seed)
    a0 = np.array([0.0])
    a = rng.normal(0.0, 1.0, p)
    w = rng.normal(0.0, 1.0, p)
    b = -0.5 * w
    return np.concatenate([a0, a, w, b])

In [12]:
def finite_difference_gradient(func, theta, eps=1e-6):
    grad = np.zeros_like(theta)
    for i in range(len(theta)):
        e_i = np.zeros_like(theta)
        e_i[i] = 1.0
        term = func(theta - 2*eps*e_i) - 8*func(theta - eps*e_i) + 8*func(theta + eps*e_i) - func(theta + 2*eps*e_i)
        grad[i] = term / (12 * eps)
    return grad

In [20]:
def C0_adam(theta_init, eM, q, lam=0.0, w_Q=1.0, w_penalty=1.0, lr=1e-3, eps=1e-6, max_iter=100):
    theta = torch.tensor(theta_init, dtype=torch.float64)
    optimizer = torch.optim.Adam([theta], lr=lr)
    loss_fn = lambda theta_np: loss_C0(theta_np, eM, q, lam=lam, w_Q=w_Q, w_penalty=w_penalty)
    best_loss = np.inf
    best_theta = None
    
    for iteration in range(max_iter):
        theta_np = np.asarray(
                    theta.detach().cpu().tolist(),
                    dtype=np.float64,
                )
        loss = loss_fn(theta_np)
        if loss < best_loss:
            best_loss = loss
            best_theta = theta_np.copy()
        
        grad_np = finite_difference_gradient(loss_fn, theta_np, eps=eps)
        optimizer.zero_grad(set_to_none=True)
        theta.grad = torch.tensor(
            grad_np.tolist(),
            dtype=theta.dtype,
            device=theta.device,
        )
        optimizer.step()
        
        if iteration % 20 == 0:
            print(
                f"step = {iteration:4d}, "
                f"loss = {loss:.6e}, "
                f"|gradient| = {np.linalg.norm(grad_np):.6e}"
            )
    return best_theta, best_loss

In [51]:
p = 2
theta0 = init_theta(p, seed=0)

theta_best, loss_best = C0_adam(
    theta_init=theta0,
    eM=6.9,
    q=1.0,
    lam=0.0,
    max_iter=500,
    lr=1e-3,
)

step =    0, loss = 9.999744e-01, |gradient| = 1.439602e-03
step =   20, loss = 9.996439e-01, |gradient| = 2.388069e-02
step =   40, loss = 9.980780e-01, |gradient| = 6.009363e-02
step =   60, loss = 9.940825e-01, |gradient| = 1.083573e-01
step =   80, loss = 9.861913e-01, |gradient| = 1.693044e-01
step =  100, loss = 9.727413e-01, |gradient| = 2.430214e-01
step =  120, loss = 9.520855e-01, |gradient| = 3.280564e-01
step =  140, loss = 9.229247e-01, |gradient| = 4.207232e-01
step =  160, loss = 8.846854e-01, |gradient| = 5.150026e-01
step =  180, loss = 8.378144e-01, |gradient| = 6.032914e-01
step =  200, loss = 7.838569e-01, |gradient| = 6.778462e-01
step =  220, loss = 7.252742e-01, |gradient| = 7.319423e-01
step =  240, loss = 6.651401e-01, |gradient| = 7.599727e-01
step =  260, loss = 6.067825e-01, |gradient| = 7.589881e-01
step =  280, loss = 5.531771e-01, |gradient| = 7.314660e-01
step =  300, loss = 5.062780e-01, |gradient| = 6.847247e-01
step =  320, loss = 4.667842e-01, |gradi

In [52]:
xi_vals, xip_vals, I = xi_integral(theta_best, k=0)

out = sup_rU_for_q_lambda_massive(
    theta=theta_best,
    k=0,
    xi_vals=xi_vals,
    xip_vals=xip_vals,
    q=1,
    lam=0,
    m_over_e=0.0,
    eM=20.0
)

print("best loss:", loss_best)
print("charge residual:", out["charge_residual"])
print("inf xi:", np.min(xi_vals))
print("sup rU:", out["sup_rU"])

best loss: 0.32850188038974765
charge residual: 0.3838710248454331
inf xi: 0.39822028233680823
sup rU: -0.16933633154756955


In [44]:
def scale_theta(theta, eM, q, lam=0.0):
    theta = np.asarray(theta, dtype=float)
    p = (len(theta) - 1) // 3

    r_plus, Qtarget = rplus_and_Qtarget(q, lam)

    def theta_at_scale(scale):
        theta_scaled = theta.copy()
        theta_scaled[:p + 1] *= scale  # scale a0, a1, ..., ap only
        return theta_scaled

    def f(scale):
        theta_scaled = theta_at_scale(scale)
        _, _, I = xi_integral(theta_scaled, k=0)

        return eM * r_plus**2 * I - Qtarget

    scale_opt = opt.root_scalar(
        f,
        bracket=[0.1, 10.0],
        method="brentq",
        xtol=1e-12,
    )

    if not scale_opt.converged:
        raise RuntimeError("Failed to find scaling factor for theta.")

    return theta_at_scale(scale_opt.root)

In [54]:
def scale_and_validate(theta, eM, q, lam=0.0):
    scaled_theta = scale_theta(theta, eM, q, lam=lam)
    xi_vals, xip_vals, I = xi_integral(scaled_theta, k=0)
    out = sup_rU_for_q_lambda_massive(
        theta=scaled_theta,
        k=0,
        xi_vals=xi_vals,
        xip_vals=xip_vals,
        q=q,
        lam=lam,
        m_over_e=0.0,
        eM=eM
    )
    if np.min(xi_vals) <= 1e-8:
        print(scaled_theta)
        raise ValueError("xi_vals negative or zero.")
    if out["sup_rU"] > 1e-6:
        raise ValueError("sup_rU positive.")
    
    return scaled_theta, xi_vals, xip_vals, I, out

In [55]:
try:
    scaled_theta, xi_vals, xip_vals, I, out = scale_and_validate(theta_best, eM=6.9, q=1.0, lam=0.0)
    print("Scaled theta validated successfully.")
    print("inf xi:", np.min(xi_vals))
    print("sup rU:", out["sup_rU"])
    print("charge residual:", out["charge_residual"])
    print("theta:", scaled_theta)
except ValueError as e:
    print("Validation failed:", e)

[-1.39702252  1.93564265 -1.8478167   0.04426563  0.73018826 -0.97218938
  0.62202684]
Validation failed: xi_vals negative or zero.
